[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frhack/oli_ai/blob/main/notebooks/oli_ai_mnist_rete_neurale_didattica.ipynb)

# Una rete neurale impara MNIST

**Punto di partenza.** Nel notebook *MNIST con similarità coseno* abbiamo riconosciuto cifre scritte a mano usando 10 immagini-prototipo e il prodotto scalare normalizzato. Accuratezza sul test set: **~82%**. Un risultato dignitoso ma chiaramente limitato — il modello era pochissimo più sofisticato di un confronto pixel-per-pixel.

**Cosa abbiamo imparato dalle porte logiche.** Nei tre notebook sulle porte (OR lineare, NOR affine, XOR con attivazione) abbiamo visto che senza **non-linearità** una rete profonda è matematicamente equivalente a un singolo neurone affine. Per imparare problemi più complicati serve introdurre tra uno strato e l'altro una funzione di attivazione non lineare — la più semplice è la **ReLU**.

**Cosa facciamo oggi.** Costruiamo una **vera rete neurale** per MNIST: tre strati (784 → 128 → 64 → 10), ReLU come attivazione, training con gradient descent. Vedremo che l'accuratezza fa un salto significativo rispetto al baseline cosine (dall'82% a oltre il 93%). Alla fine c'è di nuovo il pannello-disegno, ma questa volta il riconoscitore sarà la **rete addestrata**.

## Setup

In [ ]:
!pip install --upgrade --no-cache-dir oli_ai > /dev/null

import jax
import jax.numpy as jnp
import jax.random as jr
from jax import grad, jit
from jax.nn import relu, one_hot
import numpy as np
import matplotlib.pyplot as plt
from oli_ai.mnist_lib import carica_mnist, show_nn_graph, mostra_cifra

## 1. I dati — MNIST

Stessi dati del notebook precedente: **60.000** immagini di training e **10.000** di test, ognuna una matrice 28×28 di pixel da 0 (bianco) a 255 (nero).

Per dare i dati in pasto a una rete neurale facciamo due trasformazioni:

1. **Appiattiamo** ogni immagine in un vettore di 784 numeri (concateniamo le righe — è la stessa idea già vista nel notebook cosine).
2. **Normalizziamo** i pixel da 0..255 a 0..1 dividendo per 255. È un'abitudine standard: lavorare con numeri piccoli e di scala simile rende il training più stabile.

In [ ]:
immagini_train, etichette_train, immagini_test, etichette_test = carica_mnist()

X_train = jnp.asarray(immagini_train.reshape(-1, 784).astype('float32') / 255.0)
X_test  = jnp.asarray(immagini_test.reshape(-1, 784).astype('float32') / 255.0)
y_train = jnp.asarray(etichette_train)
y_test  = jnp.asarray(etichette_test)

print(f'X_train shape: {X_train.shape}    y_train shape: {y_train.shape}')
print(f'X_test  shape: {X_test.shape}     y_test  shape: {y_test.shape}')

# Vediamo una cifra di esempio
mostra_cifra(immagini_train[0])
print(f'etichetta: {int(y_train[0])}')

## 2. L'architettura della rete

Costruiamo una rete con tre strati:

- **input**: 784 neuroni (uno per pixel)
- **primo strato nascosto**: 128 neuroni
- **secondo strato nascosto**: 64 neuroni
- **output**: 10 neuroni (uno per cifra)

Tra uno strato e l'altro applichiamo la funzione di attivazione **ReLU** (la rettificazione `max(0, x)` che abbiamo conosciuto nel notebook XOR).

In [ ]:
show_nn_graph([784, 128, 64, 10], max_neurons=8)

# Conteggio dei parametri della rete
# - 784 -> 128: W1 (784,128) = 100352 pesi, b1 (128,) = 128 bias
# - 128 -> 64:  W2 (128, 64) =   8192 pesi, b2  (64,) =  64 bias
# - 64  -> 10:  W3 (64,  10) =    640 pesi, b3  (10,) =  10 bias
pesi  = 784*128 + 128*64 + 64*10
bias  = 128 + 64 + 10
print(f'Neuroni:    {784 + 128 + 64 + 10}')
print(f'Pesi:       {pesi:,}')
print(f'Bias:       {bias:,}')
print(f'Parametri:  {pesi + bias:,}')

## 3. Il modello (con attivazione ReLU)

La funzione `predict` calcola il forward pass della rete in tre passi:

1. Primo strato:  `h1 = ReLU(X · W1 + b1)`  →  (N, 128)
2. Secondo strato: `h2 = ReLU(h1 · W2 + b2)` →  (N, 64)
3. Output:         `out = h2 · W3 + b3`     →  (N, 10)

L'output ha 10 numeri per immagine: ognuno è un "punteggio" per una cifra. La cifra predetta sarà quella con il punteggio più alto (`argmax`).

In [ ]:
attivazione = relu

def predict(params, X):
    W1, b1, W2, b2, W3, b3 = params
    h1 = attivazione(X @ W1 + b1)
    h2 = attivazione(h1 @ W2 + b2)
    return h2 @ W3 + b3

## 4. La loss

Per il training abbiamo bisogno di una "risposta giusta" che sia confrontabile con l'output della rete (10 numeri per immagine). Codifichiamo le etichette in **one-hot**: l'etichetta `3` diventa il vettore `[0,0,0,1,0,0,0,0,0,0]`, l'etichetta `7` diventa `[0,0,0,0,0,0,0,1,0,0]`, e così via.

La loss è il **MSE** (errore quadratico medio) tra l'output della rete e il vettore one-hot — la stessa loss che abbiamo usato nei notebook delle porte logiche, solo applicata su 10 numeri invece che su 1.

In [ ]:
y_train_oh = one_hot(y_train, 10)
print(f'y_train_oh shape: {y_train_oh.shape}')
print(f'esempio: la cifra {int(y_train[0])} diventa {y_train_oh[0]}')

def loss(params, X, y_oh):
    return jnp.mean((y_oh - predict(params, X)) ** 2)

## 5. Inizializzazione dei parametri

Come nel notebook XOR, **non possiamo partire da zero** — i neuroni di ogni strato sono tutti uguali e i gradienti restano bloccati. Inizializziamo con piccoli numeri casuali.

Per le reti con ReLU c'è una formula standard che funziona molto bene, chiamata **inizializzazione di He**: i pesi si estraggono da una distribuzione normale con deviazione standard $\sqrt{2/n_\text{input}}$, dove $n_\text{input}$ è il numero di input di quello strato. È un dettaglio tecnico, ma rende il training molto più rapido e stabile.

In [ ]:
key = jr.PRNGKey(42)
k1, k2, k3 = jr.split(key, 3)

W1 = jr.normal(k1, (784, 128)) * jnp.sqrt(2/784)
b1 = jnp.zeros(128)
W2 = jr.normal(k2, (128, 64))  * jnp.sqrt(2/128)
b2 = jnp.zeros(64)
W3 = jr.normal(k3, (64, 10))   * jnp.sqrt(2/64)
b3 = jnp.zeros(10)

params = [W1, b1, W2, b2, W3, b3]
print(f'parametri totali: {sum(p.size for p in params):,}')
print(f'L iniziale: {float(loss(params, X_train, y_train_oh)):.4f}')

## 6. Il gradiente e il loop di addestramento

Il `grad` di JAX calcola le derivate parziali della loss rispetto a ognuno dei 109.386 parametri della rete. Una volta che abbiamo i gradienti, l'aggiornamento è uguale a quello del notebook XOR: per ogni parametro, ci spostiamo nella direzione opposta al suo gradiente, di un passo proporzionale al learning rate `lr`.

Usiamo `jit` per compilare la funzione di gradiente — con 109 mila parametri e 60 mila immagini, senza `jit` ogni epoca sarebbe lentissima.

In [ ]:
grad_loss = jit(grad(loss))
loss_jit  = jit(loss)             # anche la loss compilata, per velocita'

lr       = 1.0
n_epoche = 500
storia_loss = []

for epoca in range(n_epoche):
    perdita = loss_jit(params, X_train, y_train_oh)
    g       = grad_loss(params, X_train, y_train_oh)
    storia_loss.append(float(perdita))

    if epoca % 50 == 0 or epoca == n_epoche - 1:
        print(f'epoca {epoca:>3}: loss = {float(perdita):.4f}')

    # aggiornamento parametri (uguale al notebook XOR)
    for i in range(len(params)):
        params[i] = params[i] - lr * g[i]

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(storia_loss, color='#38bdf8')
plt.xlabel('epoca'); plt.ylabel('loss'); plt.title('Andamento della loss durante il training')
plt.grid(alpha=0.3); plt.show()

## 7. Le predizioni e l'accuratezza

Per fare una predizione su un'immagine: passiamo `X` alla rete, otteniamo 10 punteggi (uno per cifra), e prendiamo l'`argmax`. Misuriamo l'accuratezza sul training e sul test set.

In [ ]:
def predici_classi(params, X):
    return jnp.argmax(predict(params, X), axis=1)

def accuratezza(params, X, y):
    return float(jnp.mean(predici_classi(params, X) == y))

acc_train = accuratezza(params, X_train, y_train)
acc_test  = accuratezza(params, X_test,  y_test)
print(f'Accuratezza training set: {acc_train*100:.2f}%')
print(f'Accuratezza test set:     {acc_test*100:.2f}%')

## 8. Confronto con la cosine similarity

| Modello | Parametri | Accuratezza test |
|---|---|---|
| Cosine similarity (10 medie) | 7.840 | ~82% |
| Rete neurale 784-128-64-10 + ReLU | **109.386** | ~93% |

La rete neurale ha **~14 volte più parametri** del baseline cosine. In cambio guadagna molti punti di accuratezza. La differenza fondamentale: il baseline cosine confronta i pixel grezzi con un prototipo, mentre la rete **impara feature intermedie** (negli strati nascosti da 128 e 64 neuroni) che descrivono le cifre in modo più astratto del pixel-per-pixel.

## 9. Prova tu — disegna una cifra

Esegui la cella qui sotto (il codice è nascosto: clicca la freccetta se vuoi guardarlo, ma non è necessario). Comparirà un pannello con un riquadro bianco: disegna dentro una cifra (0–9) e clicca **Inferisci**. La **rete neurale che hai appena addestrato** ti dirà cosa pensa di vedere, e ti mostrerà la sua "fiducia" per ognuna delle 10 cifre.

Come nel notebook cosine, prova a disegnare **grande e centrato** per ottenere predizioni più affidabili.

In [ ]:
#@title 🎨 Pannello di disegno — clicca per avviare {display-mode: "form"}

# Cella tecnica: canvas HTML/JavaScript + decoder Python + chiamata alla rete.
# Non e' necessario leggerla.

import base64
import io
from PIL import Image
from IPython.display import HTML, display, JSON
from google.colab import output
from scipy.ndimage import center_of_mass, shift as _shift


_HTML_PANNELLO = """
<style>
  #disegno-area { font-family: sans-serif; text-align: center; padding: 12px; background: #f8f8f8; border-radius: 8px; max-width: 380px; margin: 12px auto; }
  #disegno-canvas { border: 3px solid #333; background: white; cursor: crosshair; touch-action: none; display: block; margin: 10px auto; }
  #disegno-area button { padding: 10px 22px; font-size: 16px; margin: 4px; cursor: pointer; border: 2px solid #333; border-radius: 6px; background: white; }
  #disegno-area button:hover { background: #eee; }
  #disegno-risultato { font-size: 26px; font-weight: bold; margin-top: 12px; min-height: 36px; color: #333; }
  #disegno-aiuto { font-size: 12px; color: #666; margin-top: 6px; }
  #disegno-sims { margin-top: 12px; padding: 6px; }
  #disegno-sims .titolo { font-size: 13px; color: #444; margin-bottom: 4px; text-align: left; padding-left: 4px; }
  #disegno-sims .sim-row { display: flex; align-items: center; font-family: monospace; font-size: 13px; gap: 6px; margin: 2px 0; }
  #disegno-sims .sim-label { width: 16px; font-weight: bold; text-align: right; color: #555; }
  #disegno-sims .sim-bar-bg { flex: 1; height: 14px; background: #eee; border-radius: 3px; overflow: hidden; }
  #disegno-sims .sim-bar { height: 100%; background: #b0b0b0; border-radius: 3px; transition: width 0.25s; }
  #disegno-sims .sim-row.vincente .sim-bar { background: #2ecc71; }
  #disegno-sims .sim-row.vincente .sim-label { color: #2ecc71; }
  #disegno-sims .sim-row.vincente .sim-value { font-weight: bold; color: #2ecc71; }
  #disegno-sims .sim-value { width: 56px; text-align: right; color: #555; }
</style>
<div id="disegno-area">
  <p>Disegna una cifra (0-9) <b>grande e centrata</b>, poi premi <b>Inferisci</b>:</p>
  <canvas id="disegno-canvas" width="280" height="280"></canvas>
  <div>
    <button onclick="disegnoInferisci()">Inferisci</button>
    <button onclick="disegnoCancella()">Cancella</button>
  </div>
  <div id="disegno-risultato">-</div>
  <div id="disegno-sims"></div>
  <div id="disegno-aiuto">Suggerimento: occupa la maggior parte del riquadro, tratto spesso, posizione centrale.</div>
</div>
<script>
(function() {
  const canvas = document.getElementById('disegno-canvas');
  const ctx = canvas.getContext('2d');
  const sfondo = () => { ctx.fillStyle = 'white'; ctx.fillRect(0, 0, canvas.width, canvas.height); };
  sfondo();
  ctx.strokeStyle = 'black';
  ctx.lineWidth = 22;
  ctx.lineCap = 'round';
  ctx.lineJoin = 'round';
  let down = false;
  const giu = (x, y) => { down = true; ctx.beginPath(); ctx.moveTo(x, y); };
  const muovi = (x, y) => { if (down) { ctx.lineTo(x, y); ctx.stroke(); } };
  const su = () => { down = false; };
  canvas.addEventListener('mousedown', e => giu(e.offsetX, e.offsetY));
  canvas.addEventListener('mousemove', e => muovi(e.offsetX, e.offsetY));
  canvas.addEventListener('mouseup', su);
  canvas.addEventListener('mouseleave', su);
  canvas.addEventListener('touchstart', e => { e.preventDefault(); const r = canvas.getBoundingClientRect(); giu(e.touches[0].clientX - r.left, e.touches[0].clientY - r.top); });
  canvas.addEventListener('touchmove',  e => { e.preventDefault(); const r = canvas.getBoundingClientRect(); muovi(e.touches[0].clientX - r.left, e.touches[0].clientY - r.top); });
  canvas.addEventListener('touchend',   e => { e.preventDefault(); su(); });
  window.disegnoCancella = function() {
    sfondo();
    document.getElementById('disegno-risultato').innerText = '-';
    document.getElementById('disegno-sims').innerHTML = '';
  };
  window.disegnoInferisci = async function() {
    const dataUrl = canvas.toDataURL('image/png');
    const out = document.getElementById('disegno-risultato');
    const simsBox = document.getElementById('disegno-sims');
    out.innerText = '...';
    simsBox.innerHTML = '';
    const r = await google.colab.kernel.invokeFunction('notebook.disegno_predici', [dataUrl], {});
    const data = r.data['application/json'];
    const cifra = data.cifra;
    if (cifra < 0) { out.innerText = 'Disegna prima qualcosa nel riquadro'; return; }
    out.innerText = 'Cifra riconosciuta: ' + cifra;
    let html = '<div class="titolo">Fiducia della rete per ogni cifra:</div>';
    for (let i = 0; i < 10; i++) {
      const s = data.fiducia[i];
      const w = Math.max(0, Math.min(100, s * 100));
      const cls = (i === cifra) ? 'sim-row vincente' : 'sim-row';
      html += '<div class="' + cls + '"><span class="sim-label">' + i + '</span><div class="sim-bar-bg"><div class="sim-bar" style="width:' + w + '%"></div></div><span class="sim-value">' + s.toFixed(3) + '</span></div>';
    }
    simsBox.innerHTML = html;
  };
})();
</script>
"""


def pannello_disegno_rete(params, predict_fn):
    """Pannello interattivo: disegna col mouse, premi 'Inferisci', vedi
    cosa predice la rete neurale per ognuna delle 10 cifre."""

    def _predici_da_canvas(data_url):
        # 1. data URL -> array 280x280, invertito (MNIST e' bianco su nero)
        _, b64 = data_url.split(',', 1)
        img = Image.open(io.BytesIO(base64.b64decode(b64))).convert('L')
        arr = (255 - np.array(img)).astype('uint8')

        # 2. Bounding box (canvas vuoto -> sentinella -1)
        mask = arr > 30
        if not mask.any():
            return JSON({'cifra': -1, 'fiducia': []})
        rs = np.where(mask.any(axis=1))[0]
        cs = np.where(mask.any(axis=0))[0]
        cropped = arr[rs[0]:rs[-1]+1, cs[0]:cs[-1]+1]

        # 3. Resize 20x20 (preservando proporzioni) come fa MNIST originale
        h, w = cropped.shape
        if h >= w:
            new_h, new_w = 20, max(1, int(round(w * 20 / h)))
        else:
            new_h, new_w = max(1, int(round(h * 20 / w))), 20
        resized = np.array(Image.fromarray(cropped).resize((new_w, new_h), Image.BICUBIC), dtype='float64')

        # 4. Centra in 28x28 per centro di massa
        canvas28 = np.zeros((28, 28), dtype='float64')
        y0 = (28 - new_h) // 2
        x0 = (28 - new_w) // 2
        canvas28[y0:y0+new_h, x0:x0+new_w] = resized
        cy, cx = center_of_mass(canvas28)
        canvas28 = _shift(canvas28, [14 - cy, 14 - cx], cval=0)

        # 5. Normalizza 0-1, appiattisci, passa alla rete
        x_vec = (canvas28.flatten() / 255.0).astype('float32')
        x_batch = jnp.asarray(x_vec[None, :])      # (1, 784)
        scores = predict_fn(params, x_batch)[0]     # (10,)
        # I raw scores della rete sono gia' "tipo probabilita'" (allenati a
        # ~0 o ~1 dalla MSE su one-hot); li clampiamo a [0,1] per la barra.
        fiducia = jnp.clip(scores, 0.0, 1.0)
        cifra = int(jnp.argmax(scores))
        return JSON({'cifra': cifra, 'fiducia': [float(f) for f in fiducia]})

    output.register_callback('notebook.disegno_predici', _predici_da_canvas)
    display(HTML(_HTML_PANNELLO))


pannello_disegno_rete(params, predict)


## Riepilogo del percorso

Questo è l'ultimo notebook del corso. Sintesi:

| Tappa | Tecnica | Accuratezza MNIST |
|---|---|---|
| Vettori + similarità coseno (10 medie) | matematica di base, no training | ~82% |
| Porte logiche OR / NOR | un neurone con gradient descent | — (didattica) |
| Porta XOR | rete `[2,8,1]` con ReLU | — (didattica) |
| **MNIST con rete neurale (questo)** | rete `[784,128,64,10]` con ReLU + 109.386 parametri | ~93% |

L'idea è la stessa in tutto il corso: parametri appresi minimizzando una loss con gradient descent. Quello che cambia è **quanti parametri** e **quanto sono espressivi** (lineari → affini → non lineari → reti profonde). Più la rete è espressiva, più riesce a catturare strutture complesse — al costo di più parametri da imparare e training più sofisticati.

Tutto quello che vedete oggi nei modelli "famosi" (GPT, modelli di visione, sistemi di traduzione automatica) è la stessa identica idea: rete neurale + loss + gradient descent. La differenza è soltanto la **scala** — non più 109 mila parametri, ma centinaia di miliardi.